In [1]:
# CELL : ENVIRONMENT SETUP, REPRODUCIBILITY AND SPARK INIT

import os
import sys
import platform
import random
import gc
import json
import time
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.backends.cudnn as cudnn

import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

# GLOBAL SEED
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

SEED = 42
seed_everything(SEED)

# LOG EXPERIMENTAL ENVIRONMENT
print("=== SET UP ENVIRONMENT ===")
print(f"OS      : {platform.system()} {platform.release()}")
print(f"Python  : {sys.version.split(' ')[0]}")
print(f"Device  : {'GPU - ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Seed    : {SEED}")
print("\n--- LIB VERSION ---")
print(f"PySpark : {pyspark.__version__}")
print(f"Pandas  : {pd.__version__}")
print(f"Numpy   : {np.__version__}")
print(f"PyTorch : {torch.__version__}")
print(f"LightGBM: {lgb.__version__}")
print("========================================\n")

=== SET UP ENVIRONMENT ===
OS      : Linux 6.6.122+
Python  : 3.12.13
Device  : GPU - Tesla T4
Seed    : 42

--- LIB VERSION ---
PySpark : 4.0.2
Pandas  : 2.3.3
Numpy   : 2.4.6
PyTorch : 2.10.0+cu128
LightGBM: 4.6.0



In [2]:

# CELL 1: PYSPARK DATA CLEANING & SPLITTING

# 1. Configs
DATA_PATH = '/kaggle/input/datasets/pgnjams/ie212-favorita-engineered-features-27-stores/kaggle/working/engineered_features_encoded_27_stores.parquet'
TRAIN_PARQUET = '/kaggle/working/wavelet_train_clean.parquet'
VAL_PARQUET = '/kaggle/working/wavelet_val_clean.parquet'
TEST_PARQUET = '/kaggle/working/wavelet_test_clean.parquet'

TRAIN_END = '2017-07-15'
VAL_START = '2017-07-16' # Day after train end
VAL_END = '2017-07-31'
TEST_START = '2017-08-01'
TEST_END = '2017-08-15'
LOOKBACK_DAYS = 15

# 2. Init Spark
spark = SparkSession.builder \
    .appName("IE212_Wavelet_Clean_Split") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.driver.memory", "16g") \
    .getOrCreate()

print(f"Loading data from {DATA_PATH}...")
df = spark.read.parquet(DATA_PATH)

# 3. Drop unwanted columns 

cols_to_drop = [c for c in df.columns if 'kalman' in c or c in ['time_idx', 'Unnamed: 0', 'wavelet_hard_sales']]
df = df.drop(*cols_to_drop)

# 4. Zero-Masking & Clipping for ALL wavelet columns (including lags)
wavelet_cols = [c for c in df.columns if 'wavelet_soft_sales' in c]
print(f"Cleaning negative values for: {wavelet_cols}")

for w_col in wavelet_cols:
    df = df.withColumn(
        w_col,
        when(col(w_col) < 0, 0.0).otherwise(col(w_col))
    )
# 5. Lookback buffer calculation for LSTM context
val_overlap_start = (datetime.strptime(VAL_START, '%Y-%m-%d') - timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d')
test_overlap_start = (datetime.strptime(TEST_START, '%Y-%m-%d') - timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d')

print(f"Splitting strategy:")
print(f"Train: <= {TRAIN_END}")
print(f"Val: {val_overlap_start} to {VAL_END} (includes 15-day lookback)")
print(f"Test: {test_overlap_start} to {TEST_END} (includes 15-day lookback)")

# 6. Save safely to Parquet
def create_split(df_spark, start_date, end_date, save_path, name):
    if not os.path.exists(save_path):
        if start_date:
            df_split = df_spark.filter((col("date") >= start_date) & (col("date") <= end_date))
        else:
            df_split = df_spark.filter(col("date") <= end_date)
        df_split.write.mode("overwrite").parquet(save_path)
        print(f"--> Saved {name} split.")
    else:
        print(f"--> {name} split already exists. Skipping write.")

create_split(df, None, TRAIN_END, TRAIN_PARQUET, "Train")
create_split(df, val_overlap_start, VAL_END, VAL_PARQUET, "Validation")
create_split(df, test_overlap_start, TEST_END, TEST_PARQUET, "Test")

# Stop spark to free RAM for Pandas & Model Training
spark.stop()
print("Spark stopped. RAM freed.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 14:50:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Loading data from /kaggle/input/datasets/pgnjams/ie212-favorita-engineered-features-27-stores/kaggle/working/engineered_features_encoded_27_stores.parquet...


Cleaning negative values for: ['wavelet_soft_sales', 'lag_1_wavelet_soft_sales', 'lag_7_wavelet_soft_sales', 'lag_16_wavelet_soft_sales']
Splitting strategy:
Train: <= 2017-07-15
Val: 2017-07-01 to 2017-07-31 (includes 15-day lookback)
Test: 2017-07-17 to 2017-08-15 (includes 15-day lookback)


26/06/08 14:50:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


--> Saved Train split.


--> Saved Validation split.


--> Saved Test split.
Spark stopped. RAM freed.


In [3]:

# CELL 2: LOAD CLEAN SPLITS TO PANDAS RAM

print("Loading clean splits into Pandas RAM...")
train_df = pd.read_parquet('/kaggle/working/wavelet_train_clean.parquet')
val_df = pd.read_parquet('/kaggle/working/wavelet_val_clean.parquet')
test_df = pd.read_parquet('/kaggle/working/wavelet_test_clean.parquet')

# Downcast floats to save final bits of RAM before LGBM
def downcast_dtypes(df):
    float_cols = df.select_dtypes(include=['float64']).columns
    df[float_cols] = df[float_cols].astype('float32')
    int_cols = df.select_dtypes(include=['int64']).columns
    df[int_cols] = df[int_cols].astype('int32')
    return df

train_df = downcast_dtypes(train_df)
val_df = downcast_dtypes(val_df)
test_df = downcast_dtypes(test_df)

print("Đang tạo thêm đặc trưng động lượng (Momentum Features) từ Wavelet...")

def add_wavelet_momentum(df):
    # Tính độ dốc (Slope/Diff): Hôm nay sóng wavelet tăng hay giảm so với hôm qua
    df['wavelet_diff_1'] = df.groupby(['store_nbr', 'item_nbr'])['wavelet_soft_sales'].diff(1)
    
    # Tính gia tốc: Tốc độ tăng đang nhanh lên hay chậm đi
    df['wavelet_diff_2'] = df.groupby(['store_nbr', 'item_nbr'])['wavelet_diff_1'].diff(1)
    
    # Fill NaN bằng 0 do diff sinh ra
    df['wavelet_diff_1'] = df['wavelet_diff_1'].fillna(0).astype('float32')
    df['wavelet_diff_2'] = df['wavelet_diff_2'].fillna(0).astype('float32')
    return df

train_df = add_wavelet_momentum(train_df)
val_df = add_wavelet_momentum(val_df)
test_df = add_wavelet_momentum(test_df)

print(f"train set: {train_df.shape[0]} rows")
print(f"val set: {val_df.shape[0]} rows")
print(f"test set: {test_df.shape[0]} rows")
print("\nData check (Train Head):")
print(train_df[['date', 'unit_sales', 'wavelet_soft_sales']].head(3))

gc.collect()
print("Ready for LightGBM training!")

Loading clean splits into Pandas RAM...
Đang tạo thêm đặc trưng động lượng (Momentum Features) từ Wavelet...
train set: 17265253 rows
val set: 1572613 rows
test set: 1501653 rows

Data check (Train Head):
         date  unit_sales  wavelet_soft_sales
0  2016-08-01         2.0             0.00000
1  2016-08-02         5.0             0.00000
2  2016-08-03         1.0             0.02273
Ready for LightGBM training!


In [4]:

# CELL 2.5: SORTING FOR TIME-SERIES CONTEXT

print("Sorting data chronologically for LSTM/LightGBM sequences...")

# Sort all dataframes inplace to save RAM
train_df.sort_values(by=['store_nbr', 'item_nbr', 'date'], inplace=True, ignore_index=True)
val_df.sort_values(by=['store_nbr', 'item_nbr', 'date'], inplace=True, ignore_index=True)
test_df.sort_values(by=['store_nbr', 'item_nbr', 'date'], inplace=True, ignore_index=True)

print("Sorting complete!")
print("Train tail check (ensure dates are ordered):")
print(train_df[['store_nbr', 'item_nbr', 'date']].tail(3))

Sorting data chronologically for LSTM/LightGBM sequences...
Sorting complete!
Train tail check (ensure dates are ordered):
          store_nbr  item_nbr        date
17265250         27   2113914  2017-07-15
17265251         27   2114566  2017-07-15
17265252         27   2124052  2016-11-27


In [5]:
# CELL 3: PYTORCH DATALOADER FOR LSTM



# config for lstm dataloader
SEQ_LENGTH = 15
BATCH_SIZE = 1024
FEATURE_COL = 'wavelet_soft_sales'
TARGET_COL = 'unit_sales'

class salesdataset(Dataset):
    def __init__(self, df, seq_length):
        self.data = df
        self.seq_length = seq_length
        
        # Tách biệt Feature (input) và Target (output)
        self.stores = self.data['store_nbr'].values
        self.items = self.data['item_nbr'].values
        self.features = self.data[FEATURE_COL].values
        self.targets = self.data[TARGET_COL].values
        
        if 'perishable' in self.data.columns:
            self.perishable = self.data['perishable'].values
        else:
            self.perishable = np.zeros(len(self.data), dtype=np.int8)
            
        self.valid_indices = self._get_valid_indices()
        
    def _get_valid_indices(self):
        stores_start = self.stores[:-self.seq_length]
        stores_end = self.stores[self.seq_length:]
        items_start = self.items[:-self.seq_length]
        items_end = self.items[self.seq_length:]
        
        mask = (stores_start == stores_end) & (items_start == items_end)
        return np.where(mask)[0]

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        start = self.valid_indices[idx]
        end = start + self.seq_length
        
        x = self.features[start:end]
        y = self.targets[end] 
        is_perishable = self.perishable[end]
        
        x = torch.tensor(x, dtype=torch.float32).unsqueeze(-1)
        y = torch.tensor(y, dtype=torch.float32)
        is_perishable = torch.tensor(is_perishable, dtype=torch.float32)
        
        return x, y, is_perishable


print(f"\n--- Preparing dataloaders for Feature: {FEATURE_COL} | Target: {TARGET_COL} ---")

train_dataset = salesdataset(train_df, SEQ_LENGTH)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)

val_dataset = salesdataset(val_df, SEQ_LENGTH)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

test_dataset = salesdataset(test_df, SEQ_LENGTH)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)


wavelet_loaders = {
    'wavelet_soft_sales': {
        'train': train_loader, 
        'val': val_loader, 
        'test': test_loader
    }
}

print(f"-> Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")


--- Preparing dataloaders for Feature: wavelet_soft_sales | Target: unit_sales ---
-> Train batches: 15659 | Val batches: 575 | Test batches: 520


In [6]:

# CELL 4: LIGHTGBM FEATURE PREPARATION

print("\nPreparing Data for LightGBM ")

# automatically select features 
# exclude metadata and targets/leaks
EXCLUDE_COLS = ['date', 'id', 'unit_sales', 'wavelet_soft_sales', 'kalman_sales']
LGBM_FEATURES = [c for c in train_df.columns if c not in EXCLUDE_COLS]
LGBM_TARGET = 'wavelet_soft_sales'

print(f"Features used for LightGBM ({len(LGBM_FEATURES)} columns):")
print(LGBM_FEATURES)

# create LGBM native datasets for memory efficiency
print("Creating LightGBM Dataset objects")
dtrain = lgb.Dataset(train_df[LGBM_FEATURES], label=train_df[LGBM_TARGET])
dval = lgb.Dataset(val_df[LGBM_FEATURES], label=val_df[LGBM_TARGET], reference=dtrain)

print("LightGBM Datasets ready")


Preparing Data for LightGBM 
Features used for LightGBM (24 columns):
['item_nbr', 'store_nbr', 'onpromotion', 'perishable', 'dcoilwtico', 'lag_1_unit_sales', 'lag_7_unit_sales', 'lag_16_unit_sales', 'lag_1_wavelet_soft_sales', 'lag_7_wavelet_soft_sales', 'lag_16_wavelet_soft_sales', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_30', 'city_idx', 'state_idx', 'type_idx', 'cluster_idx', 'family_idx', 'class_idx', 'holiday_type_idx', 'locale_idx', 'wavelet_diff_1', 'wavelet_diff_2']
Creating LightGBM Dataset objects
LightGBM Datasets ready


In [7]:
# CELL 5: LSTM ARCHITECTURE & LOSS (GPU)

# configs
INPUT_SIZE = 1
HIDDEN_SIZE = 32
NUM_LAYERS = 1
OUTPUT_SIZE = 1

# setup device 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# define lstm feature extractor
class lstm_extractor(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        
        # extract the last hidden state for lightgbm (net features)
        # shape: (batch, hidden_size)
        last_hidden_state = out[:, -1, :] 
        
        # predict to calculate loss and update lstm weights
        prediction = self.fc(last_hidden_state)
        
        return prediction.squeeze(-1), last_hidden_state

# define custom nwrmsle loss with dynamic weights
class nwrmsle_loss(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, y_pred, y_true, is_perishable):
        # dynamic weight: 1.25 if perishable, 1.0 otherwise
        weights = 1.0 + 0.25 * is_perishable
        
        # clip to 0 to prevent nan in log1p
        y_pred = torch.clamp(y_pred, min=0.0)
        y_true = torch.clamp(y_true, min=0.0)
        
        log_pred = torch.log1p(y_pred)
        log_true = torch.log1p(y_true)
        
        squared_error = (log_pred - log_true) ** 2
        
        # added + 1e-6 to denominator for numerical stability
        loss = torch.sum(weights * squared_error) / (torch.sum(weights) + 1e-6)
        
        return torch.sqrt(loss)

# initialize model and move to GPU
print("initializing lstm model")
model = lstm_extractor(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, OUTPUT_SIZE).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"model architecture:\n{model}")
print(f"total trainable params: {total_params}")

Using device: cuda
initializing lstm model
model architecture:
lstm_extractor(
  (lstm): LSTM(1, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)
total trainable params: 4513


In [8]:
# CELL 6: LSTM TRAINING (SINGLE GPU OPTIMIZED)

EPOCHS = 5
LR = 0.001
MODEL_SAVE_PATH = 'best_lstm_wavelet.pth'

if torch.cuda.is_available(): 
    # cudnn.benchmark = True 
    print("-> CUDNN Benchmark enabled for maximum speed.")


target_name = 'wavelet_soft_sales'
loaders = wavelet_loaders[target_name]

print(f"\n# Training LSTM on signal: {target_name}")


criterion = nwrmsle_loss().to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)

best_loss = float('inf')

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0.0
    
    for i, (x, y, w) in enumerate(loaders['train']):
        x, y, w = x.to(device), y.to(device), w.to(device)
        
        optimizer.zero_grad()
        pred, _ = model(x)
        loss = criterion(pred, y, w)
        loss.backward()
        optimizer.step()
        
        # accumulate loss correctly
        train_loss += loss.item() * x.size(0)
        
        if (i + 1) % 3000 == 0:
            print(f"  [Epoch {epoch+1}] processed {i + 1}/{len(loaders['train'])} batches...")
            
    train_loss /= len(loaders['train'].dataset)
    
    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y, w in loaders['val']:
            x, y, w = x.to(device), y.to(device), w.to(device)
            pred, _ = model(x)
            val_loss += criterion(pred, y, w).item() * x.size(0)
            
    val_loss /= len(loaders['val'].dataset)
    epoch_time = time.time() - start_time
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Time: {epoch_time:.1f}s")
    
    # Save best model checkpoint
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"  -> Best model saved! (Val Loss: {best_loss:.4f})")

print(f"\n>>> TRAINING COMPLETE. Best Val Loss: {best_loss:.4f} <<<")

-> CUDNN Benchmark enabled for maximum speed.

# Training LSTM on signal: wavelet_soft_sales
  [Epoch 1] processed 3000/15659 batches...
  [Epoch 1] processed 6000/15659 batches...
  [Epoch 1] processed 9000/15659 batches...
  [Epoch 1] processed 12000/15659 batches...
  [Epoch 1] processed 15000/15659 batches...
Epoch 1/5 | Train Loss: 0.5607 | Val Loss: 0.5555 | Time: 391.9s
  -> Best model saved! (Val Loss: 0.5555)
  [Epoch 2] processed 3000/15659 batches...
  [Epoch 2] processed 6000/15659 batches...
  [Epoch 2] processed 9000/15659 batches...
  [Epoch 2] processed 12000/15659 batches...
  [Epoch 2] processed 15000/15659 batches...
Epoch 2/5 | Train Loss: 0.5340 | Val Loss: 0.5488 | Time: 474.4s
  -> Best model saved! (Val Loss: 0.5488)
  [Epoch 3] processed 3000/15659 batches...
  [Epoch 3] processed 6000/15659 batches...
  [Epoch 3] processed 9000/15659 batches...
  [Epoch 3] processed 12000/15659 batches...
  [Epoch 3] processed 15000/15659 batches...
Epoch 3/5 | Train Loss: 0.5

In [9]:

# CELL 7: HYBRID FEATURE EXTRACTION (LSTM -> LGBM)


print("Loading best LSTM weights for feature extraction...")
model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()

def extract_hybrid_features(loader, df, hidden_size):
    """Passes data through LSTM, extracts hidden state, and maps back to dataframe"""
    print(f"Extracting features for {len(loader)} batches...")
    
    # 1. create column names
    hidden_cols = [f'lstm_h_{i}' for i in range(hidden_size)]
    
    # 2. init zero columns efficiently in pandas
    for col in hidden_cols:
        df[col] = 0.0
        
    all_hidden = []
    
    # 3. extract features using GPU
    with torch.no_grad():
        for x, y, w in loader:
            x = x.to(device)
            # ignore predictions, we only want the hidden states!
            _, hidden = model(x) 
            all_hidden.append(hidden.cpu().numpy())
            
    all_hidden = np.concatenate(all_hidden, axis=0)
    
    # 4. Map back to dataframe accurately!
    # Because valid_indices drops the first 15 days, we align using start + seq_length
    target_rows = loader.dataset.valid_indices + loader.dataset.seq_length
    
    # Fast bulk assignment using .loc
    df.loc[target_rows, hidden_cols] = all_hidden
    
    # Downcast to save RAM
    df[hidden_cols] = df[hidden_cols].astype(np.float32)
    return df, hidden_cols

# Run extraction for all splits
print("\n[Train Set]")
train_df, lstm_new_cols = extract_hybrid_features(loaders['train'], train_df, HIDDEN_SIZE)

print("\n[Validation Set]")
val_df, _ = extract_hybrid_features(loaders['val'], val_df, HIDDEN_SIZE)

print("\n[Test Set]")
test_df, _ = extract_hybrid_features(loaders['test'], test_df, HIDDEN_SIZE)

# Add new neural features to LightGBM feature list
LGBM_FEATURES.extend(lstm_new_cols)

print(f"\n>>> HYBRID FUSION COMPLETE! Added {HIDDEN_SIZE} neural features to LightGBM.")
print(f"Total features for LightGBM now: {len(LGBM_FEATURES)}")
print(train_df[['date', 'unit_sales'] + lstm_new_cols[:2]].tail(3))

Loading best LSTM weights for feature extraction...

[Train Set]
Extracting features for 15659 batches...

[Validation Set]
Extracting features for 575 batches...

[Test Set]
Extracting features for 520 batches...

>>> HYBRID FUSION COMPLETE! Added 32 neural features to LightGBM.
Total features for LightGBM now: 56
                date  unit_sales  lstm_h_0  lstm_h_1
17265250  2017-07-15        14.0       0.0       0.0
17265251  2017-07-15         1.0       0.0       0.0
17265252  2016-11-27        58.0       0.0       0.0


In [10]:

# CELL 8: CHECKPOINT HYBRID DATASETS (SAVE RAM)

print("Saving fused datasets (LSTM features included) to prevent RAM crash...")

# Lưu thẳng xuống ổ đĩa Kaggle
train_df.to_parquet('/kaggle/working/wavelet_train_fused.parquet')
val_df.to_parquet('/kaggle/working/wavelet_val_fused.parquet')
test_df.to_parquet('/kaggle/working/wavelet_test_fused.parquet')

print("Checkpoints saved successfully! You are safe now.")

# Kích hoạt bộ thu gom rác dọn dẹp RAM dư thừa
gc.collect()

Saving fused datasets (LSTM features included) to prevent RAM crash...
Checkpoints saved successfully! You are safe now.


0

In [11]:

# CELL 8.5: DOWNLOAD ARTIFACTS TO LOCAL MACHINE

import shutil
import os
from IPython.display import FileLink

print("Zipping important artifacts for local download...")

# Tạo một thư mục tạm để gom các file cần thiết
backup_dir = '/kaggle/working/ie212_wavelet_backup'
os.makedirs(backup_dir, exist_ok=True)

# Copy các file quan trọng vào thư mục backup
files_to_backup = [
    '/kaggle/working/best_lstm_wavelet.pth',
    '/kaggle/working/wavelet_train_fused.parquet',
    '/kaggle/working/wavelet_val_fused.parquet',
    '/kaggle/working/wavelet_test_fused.parquet'
]

for f in files_to_backup:
    if os.path.exists(f):
        shutil.copy(f, backup_dir)
        print(f"Copied {f}")
    else:
        print(f"WARNING: File {f} not found!")

# Nén thành file zip
zip_path = '/kaggle/working/ie212_wavelet_backup_final'
shutil.make_archive(zip_path, 'zip', backup_dir)

print(f"\nZipping complete! Click the link below to download to your local machine:")
display(FileLink(r'ie212_wavelet_backup_final.zip'))

Zipping important artifacts for local download...
Copied /kaggle/working/best_lstm_wavelet.pth
Copied /kaggle/working/wavelet_train_fused.parquet
Copied /kaggle/working/wavelet_val_fused.parquet
Copied /kaggle/working/wavelet_test_fused.parquet

Zipping complete! Click the link below to download to your local machine:


/kaggle/working/ie212_wavelet_backup_final.zip

In [ ]:

# CELL 9 (FIXED TYPE): LIGHTGBM TRAINING ON TRUE SALES

import lightgbm as lgb
import numpy as np
import gc
import pandas as pd

print("\n--- Fixing Target & Memory Optimization ---")
if 'test_df' in locals():
    del test_df
    gc.collect()

EXCLUDE_COLS = ['date', 'id', 'unit_sales', 'wavelet_soft_sales', 'kalman_sales']
LGBM_FEATURES = [c for c in train_df.columns if c not in EXCLUDE_COLS]

#  LightGBM MUST predict the TRUE raw sales!
LGBM_TARGET = 'unit_sales'

y_train_lgb = np.log1p(train_df[LGBM_TARGET].clip(lower=0))
y_val_lgb = np.log1p(val_df[LGBM_TARGET].clip(lower=0))

w_train = 1.0 + 0.25 * train_df['perishable'].values
w_val = 1.0 + 0.25 * val_df['perishable'].values

dtrain = lgb.Dataset(train_df[LGBM_FEATURES], label=y_train_lgb, weight=w_train)

# Handle datetime vs string TypeError safely
val_mask = pd.to_datetime(val_df['date']) >= pd.to_datetime('2017-07-16')

dval = lgb.Dataset(
    val_df.loc[val_mask, LGBM_FEATURES], 
    label=y_val_lgb[val_mask], 
    weight=w_val[val_mask], 
    reference=dtrain
)

# params = {
#     'objective': 'regression',
#     'metric': 'rmse',
#     'learning_rate': 0.03,
#     'num_leaves': 127,
#     'min_data_in_leaf': 50,
#     'feature_fraction': 0.8,
#     'bagging_fraction': 0.8,
#     'bagging_freq': 1,
#     'n_jobs': -1,
#     'seed': SEED
# }

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.02,
    'num_leaves': 80,
    'min_data_in_leaf': 200,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.7,
    'bagging_freq': 1,
    'verbose': -1,
    'seed': SEED
}


print("Starting LightGBM training on TRUE unit_sales...")
evals_result = {}
lgb_model = lgb.train(
    params, dtrain, num_boost_round=1500,
    valid_sets=[dtrain, dval], valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50), 
        lgb.log_evaluation(50),
        lgb.record_evaluation(evals_result) 
    ]
)

lgb_model.save_model('lightgbm_wavelet_hybrid.txt')
gc.collect()


--- Fixing Target & Memory Optimization ---
Starting LightGBM training on TRUE unit_sales...
Training until validation scores don't improve for 50 rounds
[50]	train's rmse: 0.532469	val's rmse: 0.525495
[100]	train's rmse: 0.463862	val's rmse: 0.456599
[150]	train's rmse: 0.447911	val's rmse: 0.440581
[200]	train's rmse: 0.442469	val's rmse: 0.435326
[250]	train's rmse: 0.439529	val's rmse: 0.432994
[300]	train's rmse: 0.436991	val's rmse: 0.431735
[350]	train's rmse: 0.435215	val's rmse: 0.430859
[400]	train's rmse: 0.433886	val's rmse: 0.430151
[450]	train's rmse: 0.432843	val's rmse: 0.42966
[500]	train's rmse: 0.432011	val's rmse: 0.429272


In [ ]:
# CELL 9c: UTILITY - WAVELET LEARNING CURVE

train_rmse = evals_result['train']['rmse']
val_rmse = evals_result['val']['rmse']
epochs_range = range(1, len(train_rmse) + 1)

plt.figure(figsize=(10, 6))
plt.plot(epochs_range, train_rmse, label='Train RMSE', linewidth=2, color='blue')
plt.plot(epochs_range, val_rmse, label='Validation RMSE', linewidth=2, color='orange')

best_iter = lgb_model.best_iteration
plt.axvline(x=best_iter, color='red', linestyle='--', label=f'Best Iteration ({best_iter})')

plt.title('Wavelet Track: LightGBM Learning Curve (Overfitting Check)', fontsize=14, fontweight='bold')
plt.xlabel('Boosting Rounds', fontsize=12)
plt.ylabel('RMSE', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('/kaggle/working/wavelet_learning_curve.png', dpi=300)
plt.show()

In [ ]:

# CELL 10: HYBRID FEATURE IMPORTANCE ANALYSIS

print("--- 1. Standard Feature Importance ---")
# use importance_type='gain' to measure actual information contribution
ax = lgb.plot_importance(lgb_model, max_num_features=20, figsize=(10, 6), importance_type='gain')
plt.title('Top 20 Features by Information Gain')
plt.tight_layout()
plt.savefig('wavelet_lgbm_importance_top20.png', dpi=300)
plt.show()

print("\n--- 2. Grouped Feature Contribution (For Report) ---")
# extract raw importances from model directly (no need to save/load csv)
feat_names = lgb_model.feature_name()
importances = lgb_model.feature_importance(importance_type='gain')
imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances})

# categorize function tailored for our specific naming convention
def categorize_feature(feat_name):
    if 'lstm_h_' in feat_name:
        return '1. LSTM Temporal Embeddings'
    elif 'lag' in feat_name:
        return '2. Statistical Lag Features'
    elif 'roll' in feat_name or 'std' in feat_name or 'mean' in feat_name:
        return '3. Rolling Window Features'
    else:
        return '4. Basic & Categorical Features'

imp_df['feature_group'] = imp_df['feature'].apply(categorize_feature)

# aggregate and calculate percentages
grouped_imp = imp_df.groupby('feature_group')['importance'].sum().reset_index()
grouped_imp['importance_percent'] = (grouped_imp['importance'] / grouped_imp['importance'].sum()) * 100
grouped_imp = grouped_imp.sort_values(by='importance_percent', ascending=False)

# plot grouped importance
plt.figure(figsize=(10, 5))
sns.barplot(x='importance_percent', y='feature_group', data=grouped_imp, palette='magma')

plt.title('Hybrid Model: Feature Group Contribution', fontsize=14, fontweight='bold')
plt.xlabel('Relative Importance (%)', fontsize=12)
plt.ylabel('Feature Group', fontsize=12)

# add percentage text labels
for index, value in enumerate(grouped_imp['importance_percent']):
    plt.text(value + 0.5, index, f'{value:.1f}%', va='center')

plt.xlim(0, max(grouped_imp['importance_percent']) + 15)
plt.tight_layout()
plt.savefig('wavelet_grouped_feature_importance.png', dpi=300)
plt.show()

print("\n--- SUMMARY TABLE ---")
print(grouped_imp[['feature_group', 'importance_percent']])

In [ ]:
# CELL 10.5: TẠO FEATURE IMPORTANCE
imp_df = pd.DataFrame({
    'feature': lgb_model.feature_name(), 
    'importance': lgb_model.feature_importance(importance_type='split')
})
# top 50 như 
imp_df.sort_values('importance', ascending=False).head(50).to_csv('/kaggle/working/track2_feature_importance.csv', index=False)
print("Saved 'track2_feature_importance.csv' (Top 50, importance_type='split')")

In [ ]:

# CELL 11: INFERENCE 

print("--- 1. Loading & Filtering Test Dataset ---")
test_df = pd.read_parquet('/kaggle/working/wavelet_test_fused.parquet')

# CRITICAL FIX: Cut off the 15-day lookback buffer with safe datetime conversion!
test_mask = pd.to_datetime(test_df['date']) >= pd.to_datetime('2017-08-01')
test_df = test_df[test_mask].reset_index(drop=True)

EXCLUDE_COLS = ['date', 'id', 'unit_sales', 'wavelet_soft_sales', 'kalman_sales']
LGBM_FEATURES = [c for c in test_df.columns if c not in EXCLUDE_COLS]

print(f"Predicting on {len(test_df)} true Kaggle test rows...")
log_preds = lgb_model.predict(test_df[LGBM_FEATURES])
preds = np.expm1(np.clip(log_preds, a_min=0, a_max=None))

print("--- 2. Saving")
submission = test_df[['id']].copy()
submission['unit_sales'] = preds
submission.to_csv('/kaggle/working/wavelet_hybrid_submission.csv', index=False)

print(">>> KAGGLE SUBMISSION SAVED <<<")
print(submission.head())

In [ ]:

# CELL 12: STREAMLIT EXPORT & TRUE LOCAL METRICS
import joblib


print("--- 1. Generating Streamlit Prediction File (Track 2) ---")
# Streamlit Export (YYYY-MM-DD, 4 decimals, no index)
streamlit_df = pd.DataFrame({
    'date': pd.to_datetime(test_df['date']).dt.strftime('%Y-%m-%d'),
    'store_nbr': test_df['store_nbr'].astype(int),
    'item_nbr': test_df['item_nbr'].astype(int),
    'actual_sales': test_df['unit_sales'].values,
    'predicted_sales': np.round(preds, 4) 
})
streamlit_df.to_csv('/kaggle/working/preds_track2.csv', index=False)
print("-> SUCCESS: Saved 'preds_track2.csv'")

print("\n--- 2. Exporting Model Weights for Streamlit ---")
joblib.dump(lgb_model, '/kaggle/working/lgb_wavelet_hybrid.pkl')
print("-> SUCCESS: Saved 'lgb_wavelet_hybrid.pkl'")

print("\n--- 3. Calculating TRUE Local Metrics ---")
actuals = streamlit_df['actual_sales'].values
predictions = streamlit_df['predicted_sales'].values
weights = 1.0 + 0.25 * test_df['perishable'].values

def nwrmsle(y_true, y_pred, w):
    log_err = (np.log1p(y_pred) - np.log1p(y_true))**2
    return np.sqrt(np.sum(w * log_err) / np.sum(w))

final_nwrmsle = nwrmsle(actuals, predictions, weights)
final_wmae = np.sum(weights * np.abs(actuals - predictions)) / np.sum(weights)
final_wrmse = np.sqrt(np.sum(weights * (actuals - predictions)**2) / np.sum(weights))

print(f"\n>>> TRUE FINAL METRICS <<<")
print(f"NWRMSLE: {final_nwrmsle:.4f}")
print(f"WMAE   : {final_wmae:.4f}")
print(f"WRMSE  : {final_wrmse:.4f}")

with open('/kaggle/working/metrics_track2.json', 'w') as f:
    json.dump({"nwrmsle": final_nwrmsle, "wmae": final_wmae, "wrmse": final_wrmse}, f)
print("-> SUCCESS: Saved 'track2_metrics.json'")

del test_df, streamlit_df
gc.collect()

In [ ]:

# CELL 12.5: REPORT ARTIFACTS & VISUALIZATIONS

print("Exporting missing Track 2 artifacts for Report documentation...")

# 1. Export top 50 feature importance to CSV
try:
    feat_names = lgb_model.feature_name()
    importances = lgb_model.feature_importance(importance_type='gain')
    imp_df = pd.DataFrame({'feature': feat_names, 'wavelet_gain': importances})
    
    # Save raw data for table documentation
    imp_df.sort_values('wavelet_gain', ascending=False).head(50).to_csv('/kaggle/working/wavelet_feature_importance.csv', index=False)
    print("-> SUCCESS: Saved 'wavelet_feature_importance.csv'")
except Exception as e:
    print(f"-> SKIPPED feature importance CSV due to error: {e}")

# 2. Export train loss history if lists exist in memory
try:
    if 'train_losses' in locals() and len(train_losses) > 0:
        df_loss = pd.DataFrame({
            'epoch': range(1, len(train_losses) + 1), 
            'wavelet_train_loss': train_losses, 
            'wavelet_val_loss': val_losses
        })
        df_loss.to_csv('/kaggle/working/wavelet_loss_history.csv', index=False)
        print("-> SUCCESS: Saved 'wavelet_loss_history.csv'")
    else:
        print("-> SKIPPED loss history CSV (lists not initialized, printed values can be used manually for report)")
except Exception as e:
    print(f"-> SKIPPED loss history due to error: {e}")

# 3. Plot Actual vs Predicted (First 200 sequences of true Kaggle test dates)
try:
    # Ensure variables exist from cell 12
    if 'actuals' in locals() and 'predictions' in locals():
        plt.figure(figsize=(12, 5))
        plt.plot(actuals[:200], label='Actual Sales (True Sales)', color='blue', alpha=0.7)
        plt.plot(predictions[:200], label='Predicted Sales (Hybrid Model)', color='orange', alpha=0.7)
        plt.title('Track 2: Actual vs Predicted Sales (Log-Haar Hybrid LSTM-LGBM)', fontsize=12, fontweight='bold')
        plt.xlabel('Sample Indices (First 200 rows)')
        plt.ylabel('Unit Sales')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.5)
        
        # Save high quality figure for the 40-page report
        plt.savefig('/kaggle/working/actual_vs_predicted_track2.png', dpi=300)
        plt.show()
        print("-> SUCCESS: Saved and plotted 'actual_vs_predicted_track2.png'")
    else:
        print("-> ERROR: 'actuals' or 'predictions' arrays not found in RAM. Run Cell 12 first.")
except Exception as e:
    print(f"-> FAILED to generate plot due to error: {e}")

In [ ]:
# ==========================================
# CELL 12.6: CROSS-TRACK SMOOTHING & SAMPLE COMPARISON (ĐÃ CHUẨN HÓA)
# ==========================================
import pandas as pd
import matplotlib.pyplot as plt

SAMPLE_STORE = 3 
SAMPLE_ITEM = 1503844
TARGET_SIGNAL = 'wavelet_soft_sales'

print(f"Extracting sample for (Store: {SAMPLE_STORE}, Item: {SAMPLE_ITEM})...")

# 1. Vẽ biểu đồ Smoothing Effect
sample_df = train_df[(train_df['store_nbr'] == SAMPLE_STORE) & (train_df['item_nbr'] == SAMPLE_ITEM)].copy()
if sample_df.empty: sample_df = val_df[(val_df['store_nbr'] == SAMPLE_STORE) & (val_df['item_nbr'] == SAMPLE_ITEM)].copy()

sample_df['date'] = pd.to_datetime(sample_df['date'])
sample_df = sample_df.sort_values('date')

plt.figure(figsize=(14, 6))
plt.plot(sample_df['date'], sample_df['unit_sales'], label='Raw Unit Sales (Actual)', alpha=0.4, color='gray')
plt.plot(sample_df['date'], sample_df[TARGET_SIGNAL], label=f'Denoised Sales ({TARGET_SIGNAL})', linewidth=2, color='blue')
plt.title(f'Wavelet Filter Smoothing Effect (Store: {SAMPLE_STORE}, Item: {SAMPLE_ITEM})', fontsize=12, fontweight='bold')
plt.legend()
plt.savefig('/kaggle/working/wavelet_smoothing_effect.png', dpi=300)
plt.show()


# Đọc file preds_track2.csv 
df_preds = pd.read_csv('/kaggle/working/preds_track2.csv')
sample_baseline = df_preds[(df_preds['store_nbr'] == 3) & (df_preds['item_nbr'] == 1503844)].copy()


sample_baseline[['date', 'actual_sales', 'predicted_sales']].to_csv('/kaggle/working/track2_sample_for_comparison.csv', index=False)

print("--> SUCCESS: Saved 'wavelet_smoothing_effect.png' and 'track2_sample_for_comparison.csv'")

In [ ]:

# UNIFIED PLOTTING LOGIC (ĐỒNG BỘ 3 TRACKS BẰNG ID CỐ ĐỊNH) - TRACK 2

import pandas as pd
import matplotlib.pyplot as plt

out_df = pd.read_csv('/kaggle/working/preds_track2.csv')

# Cố định ID 
PLOT_STORE = 3  
PLOT_ITEM = 1503844  



# Lọc dữ liệu
plot_df = out_df[(out_df['store_nbr'] == PLOT_STORE) & (out_df['item_nbr'] == PLOT_ITEM)].copy()
plot_df['date'] = pd.to_datetime(plot_df['date'])

# Vẽ biểu đồ riêng cho Track 2 
plt.figure(figsize=(12, 5))
plt.plot(plot_df['date'], plot_df['actual_sales'], label='Actual Sales', marker='o', color='black')
plt.plot(plot_df['date'], plot_df['predicted_sales'], label='Track 2 Predict', marker='x', color='red', linestyle='--')
plt.title(f'Actual vs Predicted Sales (Test Set) - Store {PLOT_STORE}, Item {PLOT_ITEM}', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Unit Sales')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/track2_forecast_plot_fixed.png', dpi=300)
plt.show()

# Lưu file CSV trung gian chờ gộp với các phương pháp khác
merge_df = plot_df[['date', 'actual_sales', 'predicted_sales']].rename(columns={'predicted_sales': 'Track2_Pred'})
merge_df.to_csv('/kaggle/working/plot_data_track2.csv', index=False)

In [ ]:
# CELL 13: EXPORT RAW DATA ARTIFACTS FOR FINAL CROSS-TRACK COMPARISON (TRACK 2)
import pandas as pd
import json
import os

print("Exporting cross-track artifacts for Track 2...")

# 1. Export feature importance scores 
importance_scores = lgb_model.feature_importance(importance_type='split')
feature_names = lgb_model.feature_name()

df_importance = pd.DataFrame({
    'feature': feature_names,
    'importance_split': importance_scores 
})

# Sort và lưu top 50 
FI_FILE = 'track2_feature_importance.csv'
df_importance = df_importance.sort_values('importance_split', ascending=False).head(50)
df_importance.to_csv(f'/kaggle/working/{FI_FILE}', index=False)
print(f"-> Đã lưu Feature Importance: /kaggle/working/{FI_FILE}")

# 2. Export final metrics 
try:
    with open('/kaggle/working/metrics_track2.json', 'r') as f:
        old_metrics = json.load(f)
        
    metrics_dict = {
        "track": "Track 2: Wavelet + LSTM + LightGBM",
        "nwrmsle": float(old_metrics.get("nwrmsle", 0)),
        "wmae": float(old_metrics.get("wmae", 0)),
        "wrmse": float(old_metrics.get("wrmse", 0))
    }

    with open('/kaggle/working/track2_metrics_final.json', 'w') as f:
        json.dump(metrics_dict, f, indent=4)
    print("-> Đã lưu Metrics mở rộng: /kaggle/working/track2_metrics_final.json")
    
except FileNotFoundError:
    print("! Cảnh báo: Không tìm thấy 'metrics_track2.json'. Hãy đảm bảo Cell 12 đã chạy thành công.")

print("\nAll Track 2 artifacts exported successfully!")

In [ ]:
# CELL 14: EXPORT, ARCHIVE & PREPARE FOR DOWNLOAD (TRACK 2)
import os
import zipfile
from IPython.display import FileLink

# Danh sách các file cần nén cho Track 2 (Đã chuẩn hóa tên)
files_to_zip = [
    'track2_metrics_final.json',           
    'track2_feature_importance.csv',       
    'preds_track2.csv',
    'plot_data_track2.csv',
    'track2_forecast_plot_fixed.png',
    'wavelet_smoothing_effect.png',        
    'lightgbm_wavelet_hybrid.txt',
    'wavelet_learning_curve.png'
]

zip_filename = 'track2_results_archive.zip'

print(f"Đang nén các file vào {zip_filename}...")
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf: 
    for file in files_to_zip:
        file_path = f'/kaggle/working/{file}'
        if os.path.exists(file_path):
            zipf.write(file_path, arcname=file)
            print(f" + Added: {file}")
        else:
            print(f" - WARNING: File not found, skipped -> {file}")

print("\nĐã nén xong!")
display(FileLink(zip_filename))